### Install Required Packages

In [36]:
#r "nuget: CsvHelper"
#r "nuget: Plotly.NET"

Installed Packages CsvHelper, 33.1.0 Plotly.NET, 5.1.0

### Using Statements

In [37]:
using CsvHelper;
using System.Globalization;
using System.IO;
using Plotly.NET;

### Define Data Model

In [38]:
public class Sale
{
    public int OrderId { get; set; }
    public string Customer { get; set; }
    public string Product { get; set; }
    public int Quantity { get; set; }
    public double Price { get; set; }
    public DateTime Date { get; set; }
}

### Create Sample Data (Embedded CSV)

In [39]:
var csvData = @"OrderId,Customer,Product,Quantity,Price,Date
1,Ali,Laptop,1,1200,2024-01-10
2,Sara,Mouse,2,25,2024-01-12
3,John,Laptop,1,1200,2024-02-05
4,Ali,Keyboard,1,80,2024-02-10
5,Sara,Laptop,1,1200,2024-03-01
6,David,Mouse,3,25,2024-03-03";

### Load Data into List

In [40]:
var reader = new StringReader(csvData);
var csv = new CsvReader(reader, CultureInfo.InvariantCulture);

var sales = csv.GetRecords<Sale>().ToList();

sales

index value 0 Submission#36+Sale OrderId 1 Customer Ali Product Laptop Quantity 1 Price 1200 Date 2024-01-10 00:00:00Z 1 Submission#36+Sale OrderId 2 Customer Sara Product Mouse Quantity 2 Price 25 Date 2024-01-12 00:00:00Z 2 Submission#36+Sale OrderId 3 Customer John Product Laptop Quantity 1 Price 1200 Date 2024-02-05 00:00:00Z 3 Submission#36+Sale OrderId 4 Customer Ali Product Keyboard Quantity 1 Price 80 Date 2024-02-10 00:00:00Z 4 Submission#36+Sale OrderId 5 Customer Sara Product Laptop Quantity 1 Price 1200 Date 2024-03-01 00:00:00Z 5 Submission#36+Sale OrderId 6 Customer David Product Mouse Quantity 3 Price 25 Date 2024-03-03 00:00:00Z

### Add Revenue Column

In [41]:
var salesWithRevenue = sales
    .Select(s => new
    {
        s.OrderId,
        s.Customer,
        s.Product,
        s.Quantity,
        s.Price,
        s.Date,
        Revenue = s.Quantity * s.Price
    })
    .ToList();

salesWithRevenue

index value 0 { OrderId = 1, Customer = Ali, Product = Laptop, Quantity = 1, Price = 1200, Date = 1/10/2024 12:00:00 AM, Revenue = 1200 } OrderId 1 Customer Ali Product Laptop Quantity 1 Price 1200 Date 2024-01-10 00:00:00Z Revenue 1200 1 { OrderId = 2, Customer = Sara, Product = Mouse, Quantity = 2, Price = 25, Date = 1/12/2024 12:00:00 AM, Revenue = 50 } OrderId 2 Customer Sara Product Mouse Quantity 2 Price 25 Date 2024-01-12 00:00:00Z Revenue 50 2 { OrderId = 3, Customer = John, Product = Laptop, Quantity = 1, Price = 1200, Date = 2/5/2024 12:00:00 AM, Revenue = 1200 } OrderId 3 Customer John Product Laptop Quantity 1 Price 1200 Date 2024-02-05 00:00:00Z Revenue 1200 3 { OrderId = 4, Customer = Ali, Product = Keyboard, Quantity = 1, Price = 80, Date = 2/10/2024 12:00:00 AM, Revenue = 80 } OrderId 4 Customer Ali Product Keyboard Quantity 1 Price 80 Date 2024-02-10 00:00:00Z Revenue 80 4 { OrderId = 5, Customer = Sara, Product = Laptop, Quantity = 1, Price = 1200, Date = 3/1/2024 12:00:00 AM, Revenue = 1200 } OrderId 5 Customer Sara Product Laptop Quantity 1 Price 1200 Date 2024-03-01 00:00:00Z Revenue 1200 5 { OrderId = 6, Customer = David, Product = Mouse, Quantity = 3, Price = 25, Date = 3/3/2024 12:00:00 AM, Revenue = 75 } OrderId 6 Customer David Product Mouse Quantity 3 Price 25 Date 2024-03-03 00:00:00Z Revenue 75

### Total Revenue

In [42]:
var totalRevenue = salesWithRevenue.Sum(s => s.Revenue);

Console.WriteLine($"Total Revenue: €{totalRevenue}");

Total Revenue: €3805


### Best Selling Product

In [43]:
var bestProduct = sales
    .GroupBy(s => s.Product)
    .Select(g => new
    {
        Product = g.Key,
        TotalQuantity = g.Sum(x => x.Quantity)
    })
    .OrderByDescending(x => x.TotalQuantity)
    .First();

bestProduct

Product,Mouse
TotalQuantity,5


### Top 3 Customers

In [44]:
var topCustomers = salesWithRevenue
    .GroupBy(s => s.Customer)
    .Select(g => new
    {
        Customer = g.Key,
        TotalSpent = g.Sum(x => x.Revenue)
    })
    .OrderByDescending(x => x.TotalSpent)
    .Take(3)
    .ToList();

topCustomers

index value 0 { Customer = Ali, TotalSpent = 1280 } Customer Ali TotalSpent 1280 1 { Customer = Sara, TotalSpent = 1250 } Customer Sara TotalSpent 1250 2 { Customer = John, TotalSpent = 1200 } Customer John TotalSpent 1200

### Monthly Revenue

In [45]:
var monthlyRevenue = salesWithRevenue
    .GroupBy(s => new { s.Date.Year, s.Date.Month })
    .Select(g => new
    {
        Month = $"{g.Key.Year}-{g.Key.Month:D2}",
        Revenue = g.Sum(x => x.Revenue)
    })
    .OrderBy(x => x.Month)
    .ToList();

monthlyRevenue

index value 0 { Month = 2024-01, Revenue = 1250 } Month 2024-01 Revenue 1250 1 { Month = 2024-02, Revenue = 1280 } Month 2024-02 Revenue 1280 2 { Month = 2024-03, Revenue = 1275 } Month 2024-03 Revenue 1275